MARKDOWN PLACEHOLDER

In [4]:
# Import all related stuff and make Supabase connection
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd

load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

In [5]:
# Load sales table from Supabase
sales_response = supabase.table("sales").select("*").execute()
df_sales = pd.DataFrame(sales_response.data)
df_sales.head()
# df_sales.shape
# df_sales.dtypes

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [6]:
# Load roducts table from Supabase
products_response = supabase.table("products").select("*").execute()
df_products = pd.DataFrame(products_response.data)
df_products.head()
# df_products.shape
# df_products.dtypes

,product_id,product_name,category,subcategory,supplier,cost_price,retail_price,eco_certified,created_at
0,1001,Praktiline seemisnahkne tennised,jalanõusid,tossud,Soome Tehdas OY,99.21,157.49,False,2022-05-12
1,1002,Mugav linane linane kostüüm,meeste_riided,ülikonnad,Riia Stils SIA,173.56,274.78,True,2022-05-10
2,1003,Elegantne orgaaniline kleit,laste_riided,kleidid,Rakvere Tekstiil OÜ,31.83,52.04,False,2020-02-19
3,1004,Minimalistlik puuvillane tuunika,naiste_riided,pluusid,Vilma Design OÜ,29.41,41.63,False,2022-12-26
4,1005,Mugav tweed kardigan,meeste_riided,kampsunid,Nordic Fashion Group OÜ,135.70,198.29,True,2021-06-19


In [7]:
# Load customers table from Supabase
customers_response = supabase.table("customers").select("*").execute()
df_customers = pd.DataFrame(customers_response.data)
df_customers.head()
# df_customers.shape
# df_customers.dtypes

,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [8]:
# Merge Sales, Products and Customers tables into one df
df = (
    df_sales
    .merge(df_products, on='product_id', how='left')
    .merge(df_customers, on='customer_id', how='left')
)
df.head()
# df.shape
# df.dtypes

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,...,eco_certified,created_at,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,...,False,2024-09-13,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,...,True,2021-10-18,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,...,False,2024-11-21,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,...,False,2020-07-06,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,...,True,2022-11-22,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [9]:
# Puhastusraport 🧹
print("Duplikaadid:", df.duplicated(subset=['invoice_id']).sum())
# print("NULL-id:\n", df.isnull().sum())

df = df.dropna(subset=["customer_id", "sale_date", "total_price"])
#print(f"Pärast NULL-ridade eemaldamist: {len(df)} rida")

# Kuupäevade parsimine
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Eesnimi + perenimi kokku, kes teab kunas vaja läheb
df["customer_name"] = df["first_name"] + " " + df["last_name"]

# Negatiivsete total_price ridade eemaldamine
neg = df[df["total_price"] < 0]
df = df[df["total_price"] >= 0]
# print(f"Negatiivseid total_price ridu: {len(neg)}")

# Puhastusraporti printimine
print(f"Puhastatud: {df.shape[0]} müügirida, {df['customer_id'].nunique()} klienti")

Duplikaadid: 0
Puhastatud: 8950 müügirida, 2540 klienti


In [11]:
# 1. Viitekuupäev
today = pd.to_datetime('2025-02-28')

# 2. Recency — viimase ostu kuupäev ja päevade arv tänaseni
recency_df = df.groupby('customer_id')['sale_date'].max().reset_index()
recency_df['recency_days'] = (today - recency_df['sale_date']).dt.days

# 3. Frequency — ostude arv
frequency_df = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']

# 4. Monetary — kogukulutus
monetary_df = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary_value']

# 5. Liidame R, F, M ühte rfm tabelisse (pd.merge)
rfm = pd.merge(
    recency_df[['customer_id', 'recency_days']], frequency_df, on='customer_id'
)
rfm = pd.merge(rfm, monetary_df, on='customer_id')

# 6. Määrame skoorid 1-5 (pd.qcut)
# Recency: vastupidine (vähem päevi = skoor 5)
rfm['R_score'] = pd.qcut(
    rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: vajab rank(method='first') duplikaatide tõttu
rfm['F_score'] = (
    pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
    .astype(int)
)

# Monetary: suurem summa = skoor 5
rfm['M_score'] = pd.qcut(
    rfm['monetary_value'], 5, labels=[1, 2, 3, 4, 5]
).astype(int)

# 7. Arvutame koguskoori ja määrame segmendid
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']


def get_segment(score):
  if score >= 13:
    return 'VIP Champions'
  elif score >= 10:
    return 'Loyal'
  elif score >= 7:
    return 'Potential'
  elif score >= 4:
    return 'At Risk'
  else:
    return 'Lost'


rfm['Segment'] = rfm['RFM_Score'].apply(get_segment)

# 8. Trükime segmentide kokkuvõtte
summary = pd.DataFrame({
    'Arv': rfm['Segment'].value_counts(),
    'Osakaal (%)': (rfm['Segment'].value_counts(normalize=True) * 100).round(1),
})

print('=== RFM SEGMENTIDE KOKKUVÕTE ===')
print(summary)

=== RFM SEGMENTIDE KOKKUVÕTE ===
               Arv  Osakaal (%)
Segment                        
Potential      759         29.9
Loyal          679         26.7
At Risk        529         20.8
VIP Champions  455         17.9
Lost           118          4.6


In [13]:
# 1. Plotly import ja ühtne värviskeem kõigile diagrammidele
import plotly.express as px
from IPython.display import Markdown, display

# Segmendid loogilises järjekorras (parimast halvimani)
SEGMENT_ORDER = ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']

# Knaflic: rõhutame ainult olulist – VIP (sinine) ja At Risk (oranž), ülejäänud hallid
SEGMENT_COLORS = {
    'VIP Champions': '#1f4e9c',
    'Loyal':         '#8fa8cf',
    'Potential':     '#c4c4c4',
    'At Risk':       '#e8743b',
    'Lost':          '#8c8c8c',
}

def clean_layout(fig):
    """Eemaldab müra: valge taust, ilma tarbetute ruudujoonteta."""
    fig.update_layout(template='plotly_white', font=dict(size=13),
                      title_font=dict(size=18), margin=dict(t=80, l=10, r=10, b=10))
    fig.update_xaxes(showgrid=False)
    return fig

In [14]:
# Diagramm 1 – Klientide jaotus segmentide kaupa
segment_counts = (
    rfm['Segment'].value_counts()
    .reindex(SEGMENT_ORDER)
    .fillna(0).astype(int)
    .rename_axis('Segment').reset_index(name='Kliente')
)
segment_counts['Osakaal'] = (segment_counts['Kliente'] / segment_counts['Kliente'].sum() * 100).round(1)

fig1 = px.bar(
    segment_counts, x='Segment', y='Kliente',
    color='Segment', color_discrete_map=SEGMENT_COLORS,
    text=segment_counts.apply(lambda r: f"{r['Kliente']} ({r['Osakaal']}%)", axis=1),
    title='Klientide arv RFM segmentide kaupa',
    labels={'Segment': 'Kliendisegment', 'Kliente': 'Klientide arv'},
)
fig1.update_traces(textposition='outside', cliponaxis=False)
fig1.update_layout(showlegend=False)
fig1.update_yaxes(showgrid=False, showticklabels=False, title=None)
clean_layout(fig1).show()

In [15]:
# Diagramm 2 – Hajuvusdiagramm: viimane ost vs kogukulutus
# Iga punkt on üks klient. Punkti suurus = ostude arv (frequency).
fig2 = px.scatter(
    rfm, x='recency_days', y='monetary_value',
    color='Segment', size='frequency',
    category_orders={'Segment': SEGMENT_ORDER},
    color_discrete_map=SEGMENT_COLORS,
    opacity=0.7,
    title='UrbanStyle kliendisegmendid (RFM)',
    labels={'recency_days': 'Päevi viimasest ostust',
            'monetary_value': 'Kogukulutus (EUR)',
            'frequency': 'Ostude arv',
            'Segment': 'Segment'},
)
fig2.update_yaxes(showgrid=False)
clean_layout(fig2).show()

In [16]:
# Diagramm 3 – Top 10 VIP klienti kogukulutuse järgi
# Kliendi nimed puhastatud df-st (customer_id on mõlemas float)
names = df[['customer_id', 'customer_name']].drop_duplicates('customer_id')

top_vip = (
    rfm[rfm['Segment'] == 'VIP Champions']
    .nlargest(10, 'monetary_value')
    .merge(names, on='customer_id', how='left')
)
top_vip['Klient'] = top_vip['customer_name'].fillna(
    'Klient ' + top_vip['customer_id'].astype(int).astype(str)
)

fig3 = px.bar(
    top_vip.sort_values('monetary_value'),   # suurim üleval
    x='monetary_value', y='Klient', orientation='h',
    text=top_vip.sort_values('monetary_value')['monetary_value'].map(lambda v: f'{v:,.0f} €'.replace(',', ' ')),
    title='Top 10 VIP klienti kogukulutuse järgi',
    labels={'monetary_value': 'Kogukulutus (EUR)', 'Klient': 'Klient'},
    hover_data={'frequency': True, 'recency_days': True},
)
fig3.update_traces(marker_color=SEGMENT_COLORS['VIP Champions'], textposition='outside', cliponaxis=False)
fig3.update_xaxes(showticklabels=False, title=None)
fig3.update_yaxes(title=None)
clean_layout(fig3).show()

In [21]:
# Võtmenäitjad MArkole
total_customers = len(rfm)
total_revenue   = rfm['monetary_value'].sum()
seg = rfm.groupby('Segment').agg(
    kliente=('customer_id', 'count'),
    kaive=('monetary_value', 'sum'),
    keskm_kulutus=('monetary_value', 'mean'),
    keskm_recency=('recency_days', 'mean'),
    keskm_ostud=('frequency', 'mean'),
).reindex(SEGMENT_ORDER)
seg['klientide_%'] = (seg['kliente'] / total_customers * 100).round(1)
seg['kaibe_%']     = (seg['kaive'] / total_revenue * 100).round(1)
display(seg.round(1))

def eur(v):
    return f'{v:,.0f} €'.replace(',', ' ')

vip, risk, lost, pot = (seg.loc[s] for s in ['VIP Champions', 'At Risk', 'Lost', 'Potential'])
top10_share = top_vip['monetary_value'].sum() / total_revenue * 100

display(Markdown(f"""
**Kokkuvõte numbritena**
- Kliente kokku: **{total_customers}**, kogukäive **{eur(total_revenue)}**
- VIP Champions: **{int(vip.kliente)} klienti ({vip['klientide_%']}%)**, kes toovad **{vip['kaibe_%']}% käibest** ({eur(vip.kaive)}); keskmine kulutus {eur(vip.keskm_kulutus)}
- Top 10 VIP klienti üksi: **{top10_share:.1f}% käibest**
- At Risk: **{int(risk.kliente)} klienti ({risk['klientide_%']}%)**, keskmiselt **{risk.keskm_recency:.0f} päeva** viimasest ostust, nende käive kokku {eur(risk.kaive)}
- Lost: {int(lost.kliente)} klienti ({lost['klientide_%']}%), Potential: {int(pot.kliente)} klienti ({pot['klientide_%']}%)
"""))

,kliente,kaive,keskm_kulutus,keskm_recency,keskm_ostud,klientide_%,kaibe_%
Segment,,,,,,,
VIP Champions,455,1146295.2,2519.3,48.7,7.7,17.9,42.8
Loyal,679,796357.2,1172.8,145.3,3.8,26.7,29.7
Potential,759,521792.9,687.5,207.5,2.5,29.9,19.5
At Risk,529,192170.2,363.3,309.6,1.6,20.8,7.2
Lost,118,20235.1,171.5,516.9,1.0,4.6,0.8



**Kokkuvõte numbritena**
- Kliente kokku: **2540**, kogukäive **2 676 851 €**
- VIP Champions: **455 klienti (17.9%)**, kes toovad **42.8% käibest** (1 146 295 €); keskmine kulutus 2 519 €
- Top 10 VIP klienti üksi: **8.6% käibest**
- At Risk: **529 klienti (20.8%)**, keskmiselt **310 päeva** viimasest ostust, nende käive kokku 192 170 €
- Lost: 118 klienti (4.6%), Potential: 759 klienti (29.9%)


### Äritõlgendus ja soovitused Markole

#### 1. VIP programm (VIP Champions)

455 klienti ehk alla viiendiku klientidest toob 42,8% kogu käibest (1,15 miljonit €). Ainuüksi top 10 VIP klienti annavad 8,6% käibest. Need kliendid ostavad keskmiselt 7,7 korda ja viimati 49 päeva tagasi, seega on nad aktiivsed. Eesmärk on neid hoida, mitte tagasi võita.

- Eritingimused: püsiv 10% soodustus, tasuta tarne ja tagastus ning uute kollektsioonide eelmüük 48 tundi enne teisi.
- Top 10 kliendile isiklik kontaktisik ja kutse suletud ostuõhtule.

*Mõõdik:* VIP-segmendi käibeosakaal püsib vähemalt 40% juures.

#### 2. Win-back kampaania (At Risk)

529 klienti (20,8%) pole ostnud keskmiselt 310 päeva ehk üle 10 kuu. Varem tõid nad kokku 192 170 € käivet. Kui nad tagasi ei tule, liiguvad nad Lost segmenti.

- Personaalne e-kiri või SMS „Me igatseme sind“ 15% sooduskoodiga, mis kehtib 14 päeva ja põhineb kliendi varasematel ostukategooriatel.
- Kes ei reageeri, saab 7 päeva pärast meeldetuletuse.

*Mõõdik ja potentsiaal:* kui 10% At Risk klientidest (~53 klienti) teeb uue keskmise ostu (363 €), toob see umbes **19 000 €** lisakäivet.

#### 3. Uute klientide kaasamine – Nurture programm (Potential)

Potential on suurim segment: 759 klienti ehk 29,9%. Nad on ostnud keskmiselt 2,5 korda, kuid viimasest ostust on möödas 208 päeva. Loyal kliendid kulutavad keskmiselt 485 € rohkem (1 173 € vs 688 €), seega on siin kõige suurem kasvuvõimalus.

- Pärast ostu automaatne tervitusseeria (3 e-kirja 2 nädala jooksul) ja kutse lojaalsusprogrammi.
- Soodustus järgmisele ostule, mis kehtib 30 päeva.

*Mõõdik ja potentsiaal:* kui 10% Potential klientidest (~76 klienti) jõuab Loyal tasemele, annab see umbes **37 000 €** lisakäivet.